In [ ]:
# PAWN FLUX warm-session kernel (W.1) — load the model ONCE, then serve a
# work-loop against the self-hosted PostgREST instance: fast repeat images
# while the container stays alive. PAWN replaces __PAWN_PAYLOAD_B64__ with
# base64(json) before the push.
import base64, json, time, datetime, io
import requests  # preinstalled on Kaggle

payload = json.loads(base64.b64decode("__PAWN_PAYLOAD_B64__").decode())

SESSION_ID = payload["session_id"]
SESSION_TOKEN = payload["session_token"]
POSTGREST_URL = payload["postgrest_url"].rstrip("/")
POLL = int(payload.get("poll_interval", 3))

# Requests to PostgREST use the restricted `pawn_anon` Postgres role (no
# bearer/API key needed), scoped to THIS session only via row-level security
# that checks the X-Session-Token header against image_sessions.session_token
# (see postgres/schema.sql) — without it, any pawn_anon caller could read or
# write any session/job row, not just this one.
REST = POSTGREST_URL
HEADERS = {
    "Content-Type": "application/json",
    "X-Session-Token": SESSION_TOKEN,
}

def now_iso():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

def now_utc():
    return datetime.datetime.now(datetime.timezone.utc)

def parse_ts(s):
    if not s:
        return None
    try:
        return datetime.datetime.fromisoformat(s.replace("Z", "+00:00"))
    except ValueError:
        return None

# Tracks the last time ANY PostgREST call (read or write) succeeded. Used by
# the supervisor below to detect total rendezvous failure (PostgREST
# unreachable for the kernel's whole life -- dead tunnel, wrong URL, RLS
# token mismatch) and self-exit instead of burning GPU quota forever on a
# kernel that can never be seen or stopped by PAWN. Initialized to kernel
# start time, not zero, so the very first pip install has a fair window.
_last_rest_ok = time.time()

def _mark_rest_ok():
    global _last_rest_ok
    _last_rest_ok = time.time()

def _rest_patch(url, params, fields, timeout, label):
    """PATCH PostgREST, NEVER raising -- every status/heartbeat write used to
    be fire-and-forget (no response check at all) YET could still raise on a
    network error, silently killing the kernel before its own error report
    ever landed (this is the exact failure seen live: a dead dev tunnel's
    DNS error raised out of cell-1's first patch_session call). Now: one
    retry after a short pause, and a loud '[pawn]' log line on any failure --
    visible in the Kaggle kernel log even though nothing here can be shown
    to the user directly. A response body of `[]` (0 rows matched) means the
    write was accepted but didn't land on any row -- almost always an RLS
    session-token mismatch -- logged distinctly since that's silent data
    loss that unlike a network error doesn't even get a non-2xx status."""
    headers = dict(HEADERS)
    headers["Prefer"] = "return=representation"
    for attempt in (1, 2):
        try:
            r = requests.patch(url, headers=headers, params=params, json=fields, timeout=timeout)
        except Exception as e:
            print(f"[pawn] {label} PATCH failed (attempt {attempt}): {e}", flush=True)
        else:
            if r.status_code < 300:
                try:
                    body = r.json()
                except Exception:
                    body = None
                if body == []:
                    print(f"[pawn] {label} PATCH matched 0 rows (RLS/session-token mismatch?)", flush=True)
                else:
                    _mark_rest_ok()
                return
            print(f"[pawn] {label} PATCH failed (attempt {attempt}): HTTP {r.status_code} {r.text[:200]}", flush=True)
        if attempt == 1:
            time.sleep(2)
    print(f"[pawn] {label} PATCH gave up after 2 attempts", flush=True)

def get_session():
    r = requests.get(f"{REST}/image_sessions", headers=HEADERS,
                     params={"id": f"eq.{SESSION_ID}", "select": "*"}, timeout=20)
    r.raise_for_status()
    _mark_rest_ok()
    data = r.json()
    return data[0] if data else None

def patch_session(fields):
    _rest_patch(f"{REST}/image_sessions", {"id": f"eq.{SESSION_ID}"}, fields, 20, "session")

def next_job():
    r = requests.get(f"{REST}/image_jobs", headers=HEADERS,
                     params={"session_id": f"eq.{SESSION_ID}", "status": "eq.queued",
                             "order": "created_at.asc", "limit": "1", "select": "*"},
                     timeout=20)
    r.raise_for_status()
    _mark_rest_ok()
    data = r.json()
    return data[0] if data else None

def patch_job(job_id, fields):
    _rest_patch(f"{REST}/image_jobs", {"id": f"eq.{job_id}"}, fields, 30, "job")

def png_b64(image):
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

import threading

def heartbeat_during(interval=20):
    """Return a context manager that sends a session heartbeat every `interval`
    seconds in a background thread. Use it to keep the session alive while a
    long blocking call (e.g. pipe()) runs.

    Usage:
        with heartbeat_during():
            image = pipe(...).images[0]
    """
    stop_evt = threading.Event()

    def _run():
        while not stop_evt.wait(interval):
            patch_session({"heartbeat_at": now_iso()})  # never raises -- see _rest_patch

    class _Ctx:
        def __enter__(self):
            self._t = threading.Thread(target=_run, daemon=True)
            self._t.start()
            return self
        def __exit__(self, *_):
            stop_evt.set()
            self._t.join(timeout=5)

    return _Ctx()


# --- Lifelong supervisor thread ---------------------------------------------
# The model load below blocks the MAIN thread for minutes (esp. FLUX). Without
# this, a Stop requested during that window is ignored until the serve loop
# starts -- and the load's success path would then overwrite it with 'ready',
# resurrecting a session the user already stopped. This daemon runs for the
# kernel's ENTIRE life: it heartbeats every poll (so PAWN can detect a dead
# kernel even mid-warmup) and, the instant it sees a stop or the expiry, it
# hard-exits the process. Kaggle exposes no external "cancel kernel" API, so a
# cooperative self-exit is the ONLY way to actually end the run and free the GPU.
#
# Heartbeat is now unconditional every tick (not gated on the read above
# succeeding) -- patch_session itself never raises (see _rest_patch), so a
# transient get_session() failure no longer silently skips a heartbeat the
# way it used to. A second self-exit path below handles the case where
# PostgREST is unreachable for this kernel's WHOLE life (dead tunnel, wrong
# URL, RLS mismatch): after 600s (_REST_UNREACHABLE_EXIT_SECONDS below) with
# zero successful contact, the kernel can never be seen or stopped by PAWN
# anyway, so it exits itself rather than burning GPU quota until Kaggle's own
# ~12h cap. Exit code 1 (not 0) marks the Kaggle run as failed, which PAWN's
# backend kernel-status probe then reports precisely instead of the kernel
# just vanishing with no trace.
import os

_sup_stop = threading.Event()
_REST_UNREACHABLE_EXIT_SECONDS = 600

def _supervisor():
    while not _sup_stop.wait(POLL):
        _sess = None
        try:
            _sess = get_session()
        except Exception as e:
            print(f"[pawn] supervisor: get_session failed: {e}", flush=True)
        if _sess is not None:
            _status = _sess.get("status")
            _expires = parse_ts(_sess.get("expires_at"))
            if _status in ("stopping", "ended") or (_expires is not None and now_utc() >= _expires):
                patch_session({"status": "ended"})
                os._exit(0)  # abrupt by design: ends the Kaggle run, frees the GPU
        patch_session({"heartbeat_at": now_iso()})
        if time.time() - _last_rest_ok > _REST_UNREACHABLE_EXIT_SECONDS:
            print(
                f"[pawn] supervisor: no successful PostgREST contact in "
                f"{_REST_UNREACHABLE_EXIT_SECONDS}s -- this kernel can't serve or "
                "be stopped from PAWN, exiting to free the GPU.",
                flush=True,
            )
            patch_session({"status": "ended"})
            os._exit(1)

threading.Thread(target=_supervisor, daemon=True).start()


In [ ]:
patch_session({"status": "installing"})
import subprocess, sys
print("Installing FLUX dependencies...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                           "diffusers", "transformers", "accelerate",
                           "sentencepiece", "protobuf"])
except Exception as e:
    patch_session({"status": "error", "error": f"dependency install failed: {e}"})
    raise
print("Dependencies installed.")


In [ ]:
# Load FLUX ONCE (the slow ~10 min). On success announce 'ready'; on failure
# record the error on the session row and exit so PAWN surfaces it (no hang).
patch_session({"status": "loading_model"})
import os, torch
try:
    from diffusers import FluxPipeline

    input_base = "/kaggle/input"
    target_dir = None
    for root, dirs, files in os.walk(input_base):
        if "model_index.json" in files:
            target_dir = root
            break
    if not target_dir:
        raise FileNotFoundError("FLUX dataset not mounted (no model_index.json found).")

    print(f"Loading FLUX pipeline from {target_dir}...")
    # ~24GB bf16 sharded across 2x T4 (bf16 mandatory on Turing; fp16 -> NaN images).
    try:
        pipe = FluxPipeline.from_pretrained(
            target_dir, torch_dtype=torch.bfloat16, device_map="balanced",
            # Cap each T4 below its ~14.56GiB usable capacity so accelerate's
            # balanced dispatcher always leaves headroom for inference-time
            # activations instead of packing weights right up to the edge --
            # a full-to-the-brim GPU 0 was observed OOMing on the very next
            # generate call even though the model itself loaded successfully.
            max_memory={0: "13GiB", 1: "13GiB"},
            local_files_only=True,  # weights are already mounted -- skip the Hub round-trip
        )
    except Exception as e:
        print(f"balanced device_map failed ({e}); retrying with CPU offload.")
        pipe = FluxPipeline.from_pretrained(target_dir, torch_dtype=torch.bfloat16, local_files_only=True)
        pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling()
    pipe.set_progress_bar_config(disable=True)
    # Guard against resurrecting a session the user stopped mid-load:
    # without this, a Stop clicked during the (long) load would be
    # silently overwritten by 'ready' here and the kernel would serve
    # anyway. (The supervisor usually hard-exits first; this closes the race.)
    # A transient REST failure here (e.g. a dead tunnel) is NOT a model-load
    # failure -- the model above already loaded fine. Treating it as fatal
    # crashed the whole run and mislabeled the session status as "model load
    # failed" even though only this unrelated race-check call failed (seen
    # live). Skip the race-check on failure and proceed to 'ready' instead.
    try:
        _g = get_session()
    except Exception as e:
        print(f"[pawn] stop-race check failed (non-fatal, proceeding to ready): {e}", flush=True)
        _g = None
    if _g and _g.get("status") in ("stopping", "ended"):
        patch_session({"status": "ended"})
        os._exit(0)
    print("FLUX loaded; session is ready.")
    patch_session({"status": "ready", "heartbeat_at": now_iso()})
except Exception as e:
    patch_session({"status": "error", "error": f"model load failed: {e}"})
    raise


In [ ]:
# Serve loop: heartbeat, honor stop/timer/cap, generate each pending job FAST
# (model already warm) and write the PNG back into its row.
images_done = 0
while True:
    try:
        sess = get_session()
    except Exception as e:
        print(f"[pawn] serve loop: get_session failed, retrying: {e}", flush=True)
        time.sleep(POLL)
        continue
    if sess is None:
        print("session row gone; exiting")
        break
    status = sess.get("status")
    expires = parse_ts(sess.get("expires_at"))
    cap = sess.get("max_images")
    if status in ("stopping", "ended"):
        patch_session({"status": "ended"})
        print("stop requested; exiting")
        break
    if expires is not None and now_utc() >= expires:
        patch_session({"status": "ended"})
        print("timer expired; exiting")
        break
    if cap is not None and images_done >= cap:
        patch_session({"status": "ended"})
        print("image cap reached; exiting")
        break
    patch_session({"heartbeat_at": now_iso()})
    try:
        job = next_job()
    except Exception as e:
        print(f"[pawn] serve loop: next_job failed, retrying: {e}", flush=True)
        time.sleep(POLL)
        continue
    if not job:
        time.sleep(POLL)
        continue
    job_id = job["id"]
    patch_job(job_id, {"status": "running", "started_at": now_iso()})
    try:
        p = job.get("params") or {}
        init_b64 = p.get("init_image_b64")
        with heartbeat_during(interval=20):
            if init_b64:
                from diffusers import FluxImg2ImgPipeline
                from PIL import Image
                init_image = Image.open(io.BytesIO(base64.b64decode(init_b64))).convert("RGB")
                img2img_pipe = FluxImg2ImgPipeline(**pipe.components)
                image = img2img_pipe(
                    prompt=job["prompt"],
                    image=init_image,
                    strength=p.get("strength", 0.6),
                    num_inference_steps=p.get("num_inference_steps", 4),
                    guidance_scale=0.0,
                ).images[0]
            else:
                image = pipe(
                    prompt=job["prompt"],
                    num_inference_steps=p.get("num_inference_steps", 4),
                    guidance_scale=0.0,
                    max_sequence_length=256,
                    width=p.get("width", 1024),
                    height=p.get("height", 1024),
                ).images[0]
        patch_job(job_id, {"status": "done", "image_b64": png_b64(image),
                           "mime": "image/png", "via": "kaggle:flux-session",
                           "done_at": now_iso()})
        images_done += 1
        patch_session({"images_done": images_done})
    except Exception as e:
        patch_job(job_id, {"status": "error", "error": str(e), "done_at": now_iso()})